# Camada Silver — `acessos`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Técnicas aplicadas:** Conversão de Tipos (`id_aluno`, `data`, `tempo_logado`, `paginas`), Validação de Regra de Negócio (`>= 0`), Detecção de Outliers (IQR e Z-Score) e Remoção de Duplicatas.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import (
    anonymize_name_lgpd,
    detect_outliers_iqr,
    detect_outliers_zscore,
    fix_mojibake_text,
    get_silver_paths,
    load_bronze,
    normalize_id_aluno,
    parse_br_float,
    parse_date_col,
    save_silver_outputs,
)

ROOT_DIR, BRONZE_DIR, SILVER_DIR = get_silver_paths()

In [ ]:
df = load_bronze("acessos", BRONZE_DIR)
rows_in = len(df)

df["id_aluno"] = normalize_id_aluno(df["id_aluno"])
df["data"] = parse_date_col(df["data"])
df["tempo_logado"] = parse_br_float(df["tempo_logado"]).clip(lower=0)
df["paginas"] = pd.to_numeric(df["paginas"], errors="coerce").clip(lower=0).astype("Int64")

df["flag_tempo_outlier_iqr"], iqr_tempo = detect_outliers_iqr(df["tempo_logado"])
df["flag_tempo_outlier_zscore"], z_tempo = detect_outliers_zscore(df["tempo_logado"])
df["flag_paginas_outlier_iqr"], iqr_pag = detect_outliers_iqr(df["paginas"].astype(float))

dups_removed = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

log = {
    "dataset": "acessos",
    "rows_in": rows_in,
    "rows_out": int(len(df)),
    "duplicates_removed": dups_removed,
    "tempo_outliers_iqr": iqr_tempo,
    "tempo_outliers_zscore": z_tempo,
    "paginas_outliers_iqr": iqr_pag,
}
save_silver_outputs(df, log, "acessos", SILVER_DIR)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.head(5)